# LeetCode 3912 — Valid Elements in an Array bằng Python

Link đề: https://leetcode.com/problems/valid-elements-in-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Prefix Maximum, Suffix Maximum

Skeleton LeetCode:

```python
class Solution:
    def findValidElements(self, nums: List[int]) -> List[int]:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: phần tử `nums[i]` là **hợp lệ** nếu nó **lớn hơn hẳn mọi phần tử bên trái** **hoặc** lớn hơn hẳn mọi phần tử bên phải; phần tử đầu và cuối luôn hợp lệ. Trả về các phần tử hợp lệ theo thứ tự ban đầu.
2. Nắm khái niệm **prefix maximum** (max tiền tố) và **suffix maximum** (max hậu tố), cùng các "kỷ lục" (record) khi duyệt trái -> phải / phải -> trái.
3. Hiểu `all([])` là `True`: cách gọn để xử lý "mọi phần tử bên trái" khi không có phần tử nào (vị trí đầu).
4. Viết 5 cách khác nhau và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: dùng `>=` thay `>`, dùng `and` thay `or`, quên rằng phần tử đầu/cuối luôn hợp lệ, so sánh với láng giềng kề thay vì mọi phần tử một phía.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Phần tử `nums[i]` là **hợp lệ** nếu thỏa **ít nhất một** trong hai điều kiện:

- `nums[i]` **lớn hơn hẳn** mọi phần tử bên trái nó (`nums[0..i-1]`), hoặc
- `nums[i]` **lớn hơn hẳn** mọi phần tử bên phải nó (`nums[i+1..n-1]`).

Phần tử **đầu tiên** và **cuối cùng** luôn hợp lệ (một phía không có phần tử nào nên điều kiện đúng hiển nhiên). Trả về **danh sách các phần tử hợp lệ** theo thứ tự xuất hiện ban đầu.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,4,2,3,2]` | `[1,2,4,3,2]` | Đầu `1` và cuối `2` luôn hợp lệ; `2` (chỉ số 1) và `4` lớn hơn mọi phần tử bên trái; `3` lớn hơn mọi phần tử bên phải (`2`); `2` (chỉ số 3) bị loại |
| 2 | `[5,5,5,5]` | `[5,5]` | Chỉ đầu và cuối; không phần tử giữa nào lớn **hơn hẳn** |
| 3 | `[1]` | `[1]` | Một phần tử luôn hợp lệ |

Constraints: `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
nums = [1, 2, 4, 2, 3, 2]
n = len(nums)
for i in range(n):
    left_ok = all(nums[j] < nums[i] for j in range(i))
    right_ok = all(nums[j] < nums[i] for j in range(i + 1, n))
    print(f"i={i} nums[i]={nums[i]} trái-ok={left_ok!s:5} phải-ok={right_ok!s:5} -> {'hợp lệ' if left_ok or right_ok else 'loại'}")

i=0 nums[i]=1 trái-ok=True  phải-ok=False -> hợp lệ
i=1 nums[i]=2 trái-ok=True  phải-ok=False -> hợp lệ
i=2 nums[i]=4 trái-ok=True  phải-ok=True  -> hợp lệ
i=3 nums[i]=2 trái-ok=False phải-ok=False -> loại
i=4 nums[i]=3 trái-ok=False phải-ok=True  -> hợp lệ
i=5 nums[i]=2 trái-ok=False phải-ok=True  -> hợp lệ


## 3. Kiến thức nền

### 3.1. `all` và `any`

`all(iterable)` đúng khi **mọi** phần tử đúng; `any(iterable)` đúng khi **có ít nhất một** phần tử đúng. Với iterable **rỗng**: `all([]) = True` và `any([]) = False`. Tính chất này giúp phần tử đầu (không có gì bên trái) tự động thỏa "mọi phần tử bên trái nhỏ hơn".

In [2]:
print(all([]), any([]))
print(all(x < 5 for x in [1, 2, 3]), all(x < 5 for x in [1, 7]), any(x > 5 for x in [1, 7]))

True False
True False True


### 3.2. Prefix maximum (max tiền tố)

`pre[i] = max(nums[0..i])`. Phần tử `nums[i]` lớn hơn hẳn mọi phần tử bên trái khi và chỉ khi `nums[i] > pre[i-1]` (hoặc `i = 0`). Khi đó `nums[i]` là một **kỷ lục mới** của lượt duyệt từ trái.

In [3]:
nums = [1, 2, 4, 2, 3, 2]
pre, cur = [], 0
for x in nums:
    cur = max(cur, x)
    pre.append(cur)
print(pre)
print([i for i in range(len(nums)) if i == 0 or nums[i] > pre[i - 1]])

[1, 2, 4, 4, 4, 4]
[0, 1, 2]


### 3.3. Suffix maximum (max hậu tố)

`suf[i] = max(nums[i..n-1])`, tính bằng cách duyệt từ phải sang trái. `nums[i]` lớn hơn hẳn mọi phần tử bên phải khi `nums[i] > suf[i+1]` (hoặc `i = n-1`).

In [4]:
nums = [1, 2, 4, 2, 3, 2]
n = len(nums)
suf, cur = [0] * n, 0
for i in range(n - 1, -1, -1):
    cur = max(cur, nums[i])
    suf[i] = cur
print(suf)
print([i for i in range(n) if i == n - 1 or nums[i] > suf[i + 1]])

[4, 4, 4, 3, 3, 2]
[2, 4, 5]


### 3.4. Giá trị khởi tạo `0` cho "chưa có gì"

Vì `nums[i] >= 1`, ta khởi tạo `best = 0` làm "max của tập rỗng": mọi phần tử đều `> 0`, nên phần tử đầu (khi chưa có gì bên trái) tự nhiên được tính là kỷ lục. Không cần `if i == 0` đặc biệt.

In [5]:
best = 0
records = []
for x in [3, 1, 4, 4, 5]:
    if x > best:
        records.append(x)
    best = max(best, x)
print(records)

[3, 4, 5]


### 3.5. Dùng cờ `bool` đánh dấu rồi lọc

Tạo mảng `ok = [False] * n`, bật `ok[i] = True` khi thỏa một điều kiện, rồi `[x for x, f in zip(nums, ok) if f]` lấy các phần tử được bật.

In [6]:
nums = [10, 20, 30]
ok = [False, True, True]
print([x for x, f in zip(nums, ok) if f])

[20, 30]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy tưởng tượng dãy cột cao thấp trên một đường thẳng.

- Đi từ **trái sang phải**, cột nào **cao hơn mọi cột đã thấy** thì "nhìn thấy được từ bên trái".
- Đi từ **phải sang trái**, cột nào cao hơn mọi cột đã thấy thì "nhìn thấy được từ bên phải".

Phần tử hợp lệ = cột nhìn thấy được **từ ít nhất một phía**. Cột đầu luôn thấy được từ trái, cột cuối luôn thấy được từ phải.

Tính tay `[1,2,4,2,3,2]`:

| i | nums[i] | max bên trái | max bên phải | `> trái`? | `> phải`? | hợp lệ? |
|---|---|---|---|---|---|---|
| 0 | 1 | (trống) | 4 | có | không | có |
| 1 | 2 | 1 | 4 | có | không | có |
| 2 | 4 | 2 | 3 | có | có | có |
| 3 | 2 | 4 | 3 | không | không | **không** |
| 4 | 3 | 4 | 2 | không | có | có |
| 5 | 2 | 4 | (trống) | không | có | có |

Kết quả `[1, 2, 4, 3, 2]`.

In [7]:
def explain(nums):
    n = len(nums)
    res = []
    for i, x in enumerate(nums):
        left = max(nums[:i], default=0)
        right = max(nums[i + 1:], default=0)
        valid = x > left or x > right
        print(f"i={i} x={x}: max trái={left} max phải={right} -> {valid}")
        if valid:
            res.append(x)
    return res

print(explain([1, 2, 4, 2, 3, 2]))
print(explain([5, 5, 5, 5]))

i=0 x=1: max trái=0 max phải=4 -> True
i=1 x=2: max trái=1 max phải=4 -> True
i=2 x=4: max trái=2 max phải=3 -> True
i=3 x=2: max trái=4 max phải=3 -> False
i=4 x=3: max trái=4 max phải=2 -> True
i=5 x=2: max trái=4 max phải=0 -> True
[1, 2, 4, 3, 2]
i=0 x=5: max trái=0 max phải=5 -> True
i=1 x=5: max trái=5 max phải=5 -> False
i=2 x=5: max trái=5 max phải=5 -> False
i=3 x=5: max trái=5 max phải=0 -> True
[5, 5]


## 5. Cách 1 — Brute Force / baseline

Với mỗi `i`, kiểm tra trực tiếp `all(nums[j] < nums[i] for j in range(i))` (mọi phần tử bên trái nhỏ hơn) hoặc `all(nums[j] < nums[i] for j in range(i + 1, n))`. Nhờ `all([]) = True`, phần tử đầu/cuối tự động hợp lệ.

Nút thắt: mỗi `i` quét lại cả hai phía, tổng `O(n²)`. Thông tin "max bên trái" của `i` tái sử dụng được từ `i - 1`.

In [8]:
class SolutionBrute:
    def findValidElements(self, nums):
        n = len(nums)
        res = []
        for i in range(n):
            left_ok = all(nums[j] < nums[i] for j in range(i))
            right_ok = all(nums[j] < nums[i] for j in range(i + 1, n))
            if left_ok or right_ok:
                res.append(nums[i])
        return res

print(SolutionBrute().findValidElements([1, 2, 4, 2, 3, 2]), SolutionBrute().findValidElements([5, 5, 5, 5]), SolutionBrute().findValidElements([1]))

[1, 2, 4, 3, 2] [5, 5] [1]


**Độ phức tạp:** Time `O(n²)`, Space `O(n)` cho kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** "lớn hơn mọi phần tử một phía" tương đương "lớn hơn **max** của phía đó". Max tiền tố và max hậu tố tính được trong một lượt, nên mỗi phần tử kiểm tra trong `O(1)`.

- `left_ok[i]`: `nums[i] > pre[i-1]` (hoặc `i = 0`) — `nums[i]` là **kỷ lục** khi duyệt từ trái.
- `right_ok[i]`: `nums[i] > suf[i+1]` (hoặc `i = n-1`) — kỷ lục khi duyệt từ phải.

**Invariant** của lượt duyệt: `best` luôn bằng max của các phần tử đã duyệt (phía đã đi qua); `x` là kỷ lục khi `x > best`. Sau đó cập nhật `best = max(best, x)`.

Phần tử hợp lệ là hợp (OR) của hai tập kỷ lục. Vì `nums[i] >= 1 > 0`, dùng `best = 0` ban đầu thì phần tử đầu và cuối tự động là kỷ lục ở lượt tương ứng.

## 7. Các cách tiếp theo

### Cách 2 — Mảng `pre` và `suf`

Dựng `pre` và `suf` rồi kiểm tra từng `i`.

In [9]:
class SolutionPreSuf:
    def findValidElements(self, nums):
        n = len(nums)
        pre, suf = [0] * n, [0] * n
        cur = 0
        for i in range(n):
            cur = max(cur, nums[i])
            pre[i] = cur
        cur = 0
        for i in range(n - 1, -1, -1):
            cur = max(cur, nums[i])
            suf[i] = cur
        res = []
        for i in range(n):
            left_ok = i == 0 or nums[i] > pre[i - 1]
            right_ok = i == n - 1 or nums[i] > suf[i + 1]
            if left_ok or right_ok:
                res.append(nums[i])
        return res

print(SolutionPreSuf().findValidElements([1, 2, 4, 2, 3, 2]), SolutionPreSuf().findValidElements([5, 5, 5, 5]), SolutionPreSuf().findValidElements([1]))

[1, 2, 4, 3, 2] [5, 5] [1]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 3 — Hai lượt đánh dấu bằng cờ

Lượt trái -> phải: bật `ok[i]` nếu `nums[i] > best`. Lượt phải -> trái: tương tự. Không cần lưu `pre`/`suf`, chỉ một mảng cờ.

In [10]:
class SolutionFlags:
    def findValidElements(self, nums):
        n = len(nums)
        ok = [False] * n
        best = 0
        for i in range(n):                       # kỷ lục từ trái
            if nums[i] > best:
                ok[i] = True
                best = nums[i]
        best = 0
        for i in range(n - 1, -1, -1):           # kỷ lục từ phải
            if nums[i] > best:
                ok[i] = True
                best = nums[i]
        return [x for x, f in zip(nums, ok) if f]

print(SolutionFlags().findValidElements([1, 2, 4, 2, 3, 2]), SolutionFlags().findValidElements([5, 5, 5, 5]), SolutionFlags().findValidElements([1]))

[1, 2, 4, 3, 2] [5, 5] [1]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (mảng cờ), thêm `O(1)` biến.

### Cách 4 — Hai tập chỉ số rồi hợp

`left = {i : nums[i] là kỷ lục từ trái}`, `right = {i : ...phải}`, kết quả là `nums[i]` cho `i` thuộc `left | right` theo thứ tự tăng của `i`.

In [11]:
class SolutionSets:
    def findValidElements(self, nums):
        n = len(nums)
        left, right = set(), set()
        best = 0
        for i in range(n):
            if nums[i] > best:
                left.add(i)
                best = nums[i]
        best = 0
        for i in reversed(range(n)):
            if nums[i] > best:
                right.add(i)
                best = nums[i]
        return [nums[i] for i in sorted(left | right)]

print(SolutionSets().findValidElements([1, 2, 4, 2, 3, 2]), SolutionSets().findValidElements([5, 5, 5, 5]), SolutionSets().findValidElements([1]))

[1, 2, 4, 3, 2] [5, 5] [1]


**Độ phức tạp:** Time `O(n log n)` (vì `sorted`; có thể bỏ nếu tránh sắp xếp), Space `O(n)`.

### Cách 5 — Chỉ mảng `suf` + biến chạy `best` cho phía trái

Giảm một mảng: tính `suf` một lần, còn phía trái dùng biến `best` cập nhật ngay trong lượt kiểm tra cuối.

In [12]:
class SolutionSufRunning:
    def findValidElements(self, nums):
        n = len(nums)
        suf = [0] * (n + 1)                      # suf[n] = 0: tập rỗng
        for i in range(n - 1, -1, -1):
            suf[i] = max(suf[i + 1], nums[i])
        res, best = [], 0
        for i, x in enumerate(nums):
            if x > best or x > suf[i + 1]:       # kỷ lục từ trái HOẶC từ phải
                res.append(x)
            best = max(best, x)
        return res

print(SolutionSufRunning().findValidElements([1, 2, 4, 2, 3, 2]), SolutionSufRunning().findValidElements([5, 5, 5, 5]), SolutionSufRunning().findValidElements([1]))

[1, 2, 4, 3, 2] [5, 5] [1]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 5 với `nums = [1, 2, 4, 2, 3, 2]`: `suf = [4, 4, 4, 3, 3, 2, 0]` (có phần tử đệm `suf[6] = 0`).

| i | x | best (trái) | suf[i+1] (phải) | `x > best`? | `x > suf[i+1]`? | kết quả |
|---|---|---|---|---|---|---|
| 0 | 1 | 0 | 4 | có | không | thêm 1 |
| 1 | 2 | 1 | 4 | có | không | thêm 2 |
| 2 | 4 | 2 | 3 | có | có | thêm 4 |
| 3 | 2 | 4 | 3 | không | không | bỏ |
| 4 | 3 | 4 | 2 | không | có | thêm 3 |
| 5 | 2 | 4 | 0 | không | có | thêm 2 |

In [13]:
def trace_suf_running(nums):
    n = len(nums)
    suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = max(suf[i + 1], nums[i])
    print("suf =", suf)
    res, best = [], 0
    for i, x in enumerate(nums):
        a, b = x > best, x > suf[i + 1]
        print(f"i={i} x={x} best={best} suf[i+1]={suf[i + 1]}: x>best={a!s:5} x>suf={b!s:5} -> {'giữ' if a or b else 'bỏ'}")
        if a or b:
            res.append(x)
        best = max(best, x)
    return res

assert trace_suf_running([1, 2, 4, 2, 3, 2]) == [1, 2, 4, 3, 2]
assert trace_suf_running([5, 5, 5, 5]) == [5, 5]
assert trace_suf_running([1]) == [1]

suf = [4, 4, 4, 3, 3, 2, 0]
i=0 x=1 best=0 suf[i+1]=4: x>best=True  x>suf=False -> giữ
i=1 x=2 best=1 suf[i+1]=4: x>best=True  x>suf=False -> giữ
i=2 x=4 best=2 suf[i+1]=3: x>best=True  x>suf=True  -> giữ
i=3 x=2 best=4 suf[i+1]=3: x>best=False x>suf=False -> bỏ
i=4 x=3 best=4 suf[i+1]=2: x>best=False x>suf=True  -> giữ
i=5 x=2 best=4 suf[i+1]=0: x>best=False x>suf=True  -> giữ
suf = [5, 5, 5, 5, 0]
i=0 x=5 best=0 suf[i+1]=5: x>best=True  x>suf=False -> giữ
i=1 x=5 best=5 suf[i+1]=5: x>best=False x>suf=False -> bỏ
i=2 x=5 best=5 suf[i+1]=5: x>best=False x>suf=False -> bỏ
i=3 x=5 best=5 suf[i+1]=0: x>best=False x>suf=True  -> giữ
suf = [1, 0]
i=0 x=1 best=0 suf[i+1]=0: x>best=True  x>suf=True  -> giữ


In [14]:
def trace_flags(nums):
    n = len(nums)
    ok = [False] * n
    best = 0
    for i in range(n):
        if nums[i] > best:
            ok[i] = True
            best = nums[i]
    print("sau lượt trái :", ok)
    best = 0
    for i in range(n - 1, -1, -1):
        if nums[i] > best:
            ok[i] = True
            best = nums[i]
    print("sau lượt phải :", ok)
    return [x for x, f in zip(nums, ok) if f]

assert trace_flags([1, 2, 4, 2, 3, 2]) == [1, 2, 4, 3, 2]

sau lượt trái : [True, True, True, False, False, False]
sau lượt phải : [True, True, True, False, True, True]


## 9. Code nộp LeetCode

Dùng Cách 5 (một mảng `suf` + biến chạy): ngắn và `O(n)`.

In [15]:
from typing import List, Optional


class Solution:
    def findValidElements(self, nums: List[int]) -> List[int]:
        n = len(nums)
        suf = [0] * (n + 1)                 # suf[i] = max(nums[i..n-1]); suf[n] = 0 (tập rỗng)
        for i in range(n - 1, -1, -1):
            suf[i] = max(suf[i + 1], nums[i])
        res = []
        best = 0                            # max các phần tử bên trái (0 nếu chưa có, vì nums[i] >= 1)
        for i, x in enumerate(nums):
            if x > best or x > suf[i + 1]:
                res.append(x)
            best = max(best, x)
        return res


print(Solution().findValidElements([1, 2, 4, 2, 3, 2]), Solution().findValidElements([5, 5, 5, 5]), Solution().findValidElements([1]))

[1, 2, 4, 3, 2] [5, 5] [1]


## 10. Vì sao thuật toán đúng

1. **Tương đương với max.** `nums[i]` lớn hơn hẳn mọi phần tử của một tập khi và chỉ khi lớn hơn max của tập đó (với tập rỗng: đúng hiển nhiên, và ta quy ước max của tập rỗng là `0 < nums[i]`).
2. **Phía trái.** Trước khi cập nhật ở bước `i`, `best = max(nums[0..i-1])` (`0` khi `i = 0`). Nên `x > best` đúng bằng "lớn hơn hẳn mọi phần tử bên trái".
3. **Phía phải.** `suf[i+1] = max(nums[i+1..n-1])` (`0` khi `i = n-1`). Nên `x > suf[i+1]` đúng bằng "lớn hơn hẳn mọi phần tử bên phải".
4. **Hợp (OR).** Điều kiện hợp lệ là một trong hai, nên dùng `or`.
5. **Đầu và cuối luôn hợp lệ.** `i = 0`: `best = 0 < nums[0]` (vì `nums[0] >= 1`). `i = n-1`: `suf[n] = 0 < nums[n-1]`.
6. **Thứ tự.** Duyệt `i` tăng dần nên kết quả theo thứ tự ban đầu.
7. **Dấu `>` chặt.** Các phần tử bằng nhau không "lớn hơn hẳn" nhau, nên `[5,5,5,5]` chỉ giữ đầu và cuối.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- Cách 1: `O(n²)` thời gian, `O(n)` bộ nhớ (kết quả).
- Cách 2, 3, 5: `O(n)` thời gian, `O(n)` bộ nhớ (một hoặc hai mảng).
- Cách 4: `O(n log n)` thời gian do `sorted`; `O(n)` bộ nhớ.

Với `n <= 100` mọi cách đều rất nhanh; sự khác biệt chỉ có ý nghĩa nếu `n` lớn hơn. Không có best/worst khác nhau đáng kể (đều duyệt hết mảng).

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên (một phần tử, hai phần tử, tăng dần, giảm dần, toàn bằng nhau) và 500 input ngẫu nhiên đúng constraints.

In [16]:
import random

solutions = [SolutionBrute(), SolutionPreSuf(), SolutionFlags(), SolutionSets(), SolutionSufRunning(), Solution()]

fixed = [
    ([1, 2, 4, 2, 3, 2], [1, 2, 4, 3, 2]),
    ([5, 5, 5, 5], [5, 5]),
    ([1], [1]),
    ([3, 7], [3, 7]),
    ([1, 2, 3, 4], [1, 2, 3, 4]),               # tăng dần: mọi phần tử là kỷ lục từ trái
    ([4, 3, 2, 1], [4, 3, 2, 1]),               # giảm dần: mọi phần tử là kỷ lục từ phải
    ([1, 3, 1], [1, 3, 1]),
    ([2, 1, 2], [2, 2]),
    ([5, 1, 5, 1, 5], [5, 5]),
    ([1, 5, 2, 5, 1], [1, 5, 5, 1]),
    ([100, 1, 1, 100], [100, 100]),
]
for nums, expected in fixed:
    for s in solutions:
        assert s.findValidElements(list(nums)) == expected, (type(s).__name__, nums)

random.seed(39120)
for _ in range(500):
    n = random.randint(1, 100)
    hi = random.choice([2, 5, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    outs = [s.findValidElements(list(nums)) for s in solutions]
    assert all(o == outs[0] for o in outs), (nums, outs)
    out = outs[0]
    assert out[0] == nums[0] and out[-1] == nums[-1] and (n == 1 or len(out) >= 2)
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) Phần tử lớn nhất (mọi bản sao **đầu tiên từ trái** và **cuối cùng từ phải**) luôn có mặt. (2) Kết quả là dãy con (subsequence) của `nums`. (3) Đảo mảng thì kết quả đảo theo (tính đối xứng của định nghĩa).

In [17]:
random.seed(3912)
for _ in range(300):
    nums = [random.randint(1, 100) for _ in range(random.randint(1, 100))]
    out = Solution().findValidElements(nums)
    it = iter(nums)
    assert all(any(y == x for y in it) for x in out)              # out là dãy con của nums
    assert max(nums) in out
    assert Solution().findValidElements(nums[::-1]) == out[::-1]
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `>=` thay vì `>`

Đề yêu cầu **lớn hơn hẳn**. Với `[5,5,5,5]`, `>=` cho phép mọi phần tử.

In [18]:
def wrong_ge(nums):
    n = len(nums)
    res = []
    for i in range(n):
        left = all(nums[j] <= nums[i] for j in range(i))             # SAI: <=
        right = all(nums[j] <= nums[i] for j in range(i + 1, n))
        if left or right:
            res.append(nums[i])
    return res

print(wrong_ge([5, 5, 5, 5]), "!=", Solution().findValidElements([5, 5, 5, 5]))
assert wrong_ge([5, 5, 5, 5]) == [5, 5, 5, 5] and Solution().findValidElements([5, 5, 5, 5]) == [5, 5]

[5, 5, 5, 5] != [5, 5]


### Lỗi 2: dùng `and` thay vì `or`

Đề cho phép thỏa **một trong hai** phía. Dùng `and` đòi cả hai, loại nhiều phần tử hợp lệ.

In [19]:
def wrong_and(nums):
    n = len(nums)
    res = []
    for i in range(n):
        left = all(nums[j] < nums[i] for j in range(i))
        right = all(nums[j] < nums[i] for j in range(i + 1, n))
        if i in (0, n - 1) or (left and right):                       # SAI: and
            res.append(nums[i])
    return res

print(wrong_and([1, 2, 4, 2, 3, 2]), "!=", Solution().findValidElements([1, 2, 4, 2, 3, 2]))
assert wrong_and([1, 2, 4, 2, 3, 2]) == [1, 4, 2] and Solution().findValidElements([1, 2, 4, 2, 3, 2]) == [1, 2, 4, 3, 2]

[1, 4, 2] != [1, 2, 4, 3, 2]


### Lỗi 3: khởi tạo `best = nums[0]` làm mất phần tử đầu

Nếu khởi tạo `best = nums[0]` rồi duyệt từ `i = 0` với điều kiện `x > best`, phần tử đầu bị coi là không hợp lệ từ phía trái (và có thể không hợp lệ từ phía phải).

In [20]:
def wrong_init_first(nums):
    n = len(nums)
    suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = max(suf[i + 1], nums[i])
    res, best = [], nums[0]                                          # SAI: nên là 0
    for i, x in enumerate(nums):
        if x > best or x > suf[i + 1]:
            res.append(x)
        best = max(best, x)
    return res

print(wrong_init_first([3, 1, 3]), "!=", Solution().findValidElements([3, 1, 3]))
assert wrong_init_first([3, 1, 3]) == [3] and Solution().findValidElements([3, 1, 3]) == [3, 3]

[3] != [3, 3]


### Lỗi 4: chỉ so sánh với láng giềng kề

"Lớn hơn mọi phần tử một phía" khác "lớn hơn phần tử kề". Với `[3, 1, 2, 5]`, số `2` lớn hơn số kề bên trái (`1`) nhưng **không** lớn hơn mọi phần tử bên trái (có `3`) nên đáp án đúng là `[3, 5]`.

In [21]:
def wrong_neighbors(nums):
    n = len(nums)
    res = []
    for i in range(n):
        left = i == 0 or nums[i] > nums[i - 1]                       # SAI: chỉ so với phần tử kề
        right = i == n - 1 or nums[i] > nums[i + 1]
        if left or right:
            res.append(nums[i])
    return res

print(wrong_neighbors([3, 1, 2, 5]), "!=", Solution().findValidElements([3, 1, 2, 5]))
assert wrong_neighbors([3, 1, 2, 5]) == [3, 2, 5] and Solution().findValidElements([3, 1, 2, 5]) == [3, 5]

[3, 2, 5] != [3, 5]


### Lỗi 5: trả về chỉ số thay vì giá trị, hoặc quên giữ thứ tự

Đề yêu cầu trả về các **phần tử** theo **thứ tự ban đầu**. Hai lượt kỷ lục (trái rồi phải) mà nối `list_trai + list_phai` sẽ sai thứ tự và trùng lặp.

In [22]:
def wrong_concat(nums):
    n = len(nums)
    left, right = [], []
    best = 0
    for x in nums:
        if x > best:
            left.append(x)
            best = x
    best = 0
    for x in reversed(nums):
        if x > best:
            right.append(x)
            best = x
    return left + right                                               # SAI: nối hai danh sách

print(wrong_concat([1, 2, 4, 2, 3, 2]), "!=", Solution().findValidElements([1, 2, 4, 2, 3, 2]))
assert wrong_concat([1, 2, 4, 2, 3, 2]) == [1, 2, 4, 2, 3, 4]

[1, 2, 4, 2, 3, 4] != [1, 2, 4, 3, 2]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lớn hơn mọi phần tử bên trái/phải" -> **prefix max / suffix max** hoặc **kỷ lục (record)** khi duyệt một chiều (mẫu "nhìn thấy được").
- `best = 0` (khi mọi giá trị dương) hoặc `-inf` làm "max của tập rỗng" để không phải xử lý riêng phần tử đầu.
- Điều kiện hợp (OR) từ hai phía: tính hai chiều độc lập rồi gộp.
- `all([]) = True`: tận dụng để viết brute force gọn.
- Test: tất cả bằng nhau, tăng/giảm dần, đỉnh ở giữa, đáy ở giữa, độ dài 1 và 2.

## 15. Biến thể của bài

1. **Buildings With an Ocean View** (1762, Premium) và **Number of Visible People in a Queue** (1944): kỷ lục từ một phía.
2. **Leaders in an Array** (GFG): phần tử lớn hơn mọi phần tử bên phải.
3. **Trapping Rain Water** (42): prefix max & suffix max.
4. **Product of Array Except Self** (238): tiền tố/hậu tố.
5. **Count Visible Mountains / Monotonic Stack** problems: dùng ngăn xếp đơn điệu.
6. Biến thể: dùng `>=` (không cần "hơn hẳn"), hoặc yêu cầu phần tử **nhỏ hơn** mọi phần tử một phía.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | `all(...)` hai phía mỗi `i` | `O(n²)` | `O(n)` | Oracle |
| 2. `pre` + `suf` | hai mảng max | `O(n)` | `O(n)` | Dễ hình dung |
| 3. Hai lượt + cờ | kỷ lục hai chiều | `O(n)` | `O(n)` | Mẫu "nhìn thấy được" |
| 4. Hai tập chỉ số | hợp rồi `sorted` | `O(n log n)` | `O(n)` | Khi cần thao tác tập |
| 5. `suf` + biến chạy | nộp bài | `O(n)` | `O(n)` | Ngắn và nhanh |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().findValidElements([2, 5, 1, 5, 3])` và `Solution().findValidElements([9, 8, 7])` ra gì?

**Bài 2 (tính tay):** với `nums = [3, 1, 4, 1, 5, 9, 2, 6]`, liệt kê kỷ lục từ trái, kỷ lục từ phải và danh sách hợp lệ.

**Bài 3 (điền TODO):** hoàn thành `left_records(nums)` trả về các phần tử là kỷ lục từ trái (lớn hơn hẳn mọi phần tử bên trái).

**Bài 4 (tự viết lại):** viết `valid_indices(nums)` trả về **danh sách chỉ số** hợp lệ (thay vì giá trị).

**Bài 5 (mini challenge):** viết `valid_with_ge(nums)` cho phiên bản "lớn hơn **hoặc bằng**" mọi phần tử một phía và so sánh số phần tử với bản gốc.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().findValidElements([2, 5, 1, 5, 3]), Solution().findValidElements([9, 8, 7]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def left_records_todo(nums):
    res = []
    # TODO: duyệt trái -> phải, giữ best, thêm x khi x > best
    return res

[2, 5, 5, 3] [9, 8, 7]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `[9, 8, 7]` giảm dần nên mọi phần tử là kỷ lục từ phải. `[2,5,1,5,3]`: kiểm tra `5` thứ hai và `1`.

**Mức 2 (vừa):** Bài 3: dùng mẫu `best = 0`. Bài 4: giống Cách 5 nhưng `append(i)`. Bài 5: đổi `>` thành `>=` trong cả hai phía.

**Mức 3 (gần đáp án):** Bài 1: `[2, 5, 5, 3]` và `[9, 8, 7]`. Bài 2: kỷ lục trái `[3, 4, 5, 9]`, kỷ lục phải `[6, 9]`; hợp: `[3, 4, 5, 9, 6]`. Bài 5: với `[5,5,5,5]` bản `>=` cho `4` phần tử.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().findValidElements([2, 5, 1, 5, 3]) == [2, 5, 5, 3] and Solution().findValidElements([9, 8, 7]) == [9, 8, 7]
assert Solution().findValidElements([3, 1, 4, 1, 5, 9, 2, 6]) == [3, 4, 5, 9, 6]

# Bài 3
def left_records(nums):
    res, best = [], 0
    for x in nums:
        if x > best:
            res.append(x)
            best = x
    return res
assert left_records([3, 1, 4, 1, 5, 9, 2, 6]) == [3, 4, 5, 9] and left_records([5, 5]) == [5]

# Bài 4
def valid_indices(nums):
    n = len(nums)
    suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = max(suf[i + 1], nums[i])
    res, best = [], 0
    for i, x in enumerate(nums):
        if x > best or x > suf[i + 1]:
            res.append(i)
        best = max(best, x)
    return res
assert valid_indices([1, 2, 4, 2, 3, 2]) == [0, 1, 2, 4, 5] and valid_indices([5, 5, 5, 5]) == [0, 3]

# Bài 5
def valid_with_ge(nums):
    n = len(nums)
    return [nums[i] for i in range(n)
            if all(nums[j] <= nums[i] for j in range(i)) or all(nums[j] <= nums[i] for j in range(i + 1, n))]
assert valid_with_ge([5, 5, 5, 5]) == [5, 5, 5, 5]
assert len(valid_with_ge([1, 2, 4, 2, 3, 2])) >= len(Solution().findValidElements([1, 2, 4, 2, 3, 2]))
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "kỷ lục từ hai phía":**

```python
n = len(nums)
suf = [0] * (n + 1)
for i in range(n - 1, -1, -1):
    suf[i] = max(suf[i + 1], nums[i])
res, best = [], 0
for i, x in enumerate(nums):
    if x > best or x > suf[i + 1]:     # kỷ lục từ trái HOẶC từ phải
        res.append(x)
    best = max(best, x)
return res
```

**Checklist trước khi submit:**

- [ ] Dùng `>` chặt (lớn hơn **hẳn**) và `or` (một trong hai phía)?
- [ ] `best` ban đầu là `0` (không phải `nums[0]`), `suf[n] = 0` để đầu/cuối luôn hợp lệ?
- [ ] Trả về **giá trị** theo **thứ tự ban đầu** (không nối hai danh sách)?
- [ ] Đã thử `[1]`, `[5,5,5,5]`, mảng tăng dần, mảng giảm dần?
- [ ] Chữ ký đúng `findValidElements(self, nums: List[int]) -> List[int]`?